In [3]:
!pip install opencv-python

In [4]:
import cv2 as cv

In [5]:
import numpy as np

In [6]:
img = cv.imread('./index.jpg')

In [7]:
img.shape

(275, 183, 3)

In [8]:
haar_data = cv.CascadeClassifier('./data.xml')

In [9]:
while True:
    faces = haar_data.detectMultiScale(img)
    for x,y,w,h in faces:
        cv.rectangle(img,(x,y),(x+w,y+h),(255,255,0),2)
    cv.imshow('result',img)
    if cv.waitKey(2)==27:
      break
cv.destroyAllWindows()

In [10]:
video_Capture = cv.VideoCapture(0)
while True:
    flag,frame = video_Capture.read()
    if flag:
        faces = haar_data.detectMultiScale(frame)
        for x,y,w,h in faces:
            cv.rectangle(frame,(x,y),(x+w,y+h),(255,255,0),2)
        cv.imshow('result',frame)
        if cv.waitKey(2)==27:
          break
video_Capture.release()
cv.destroyAllWindows()

In [12]:
video_Capture = cv.VideoCapture(0)
data = []
while True:
    flag,frame = video_Capture.read()
    if flag:
        faces = haar_data.detectMultiScale(frame)
        for x,y,w,h in faces:
            cv.rectangle(frame,(x,y),(x+w,y+h),(255,255,0),2)
            face = frame[y:y+h, x:x+w, :]
            cv.resize(face,(50,50))
            if(len(data)<800):
                data.append(face)
        cv.imshow('result',frame)
        if cv.waitKey(2)==27 or len(data)>=400:
          break
video_Capture.release()
cv.destroyAllWindows()

In [14]:
np.save('Without_Mask.npy',data)

In [15]:
video_Capture = cv.VideoCapture(0)
data = []
while True:
    flag,frame = video_Capture.read()
    if flag:
        faces = haar_data.detectMultiScale(frame)
        for x,y,w,h in faces:
            cv.rectangle(frame,(x,y),(x+w,y+h),(255,255,0),2)
            face = frame[y:y+h, x:x+w, :]
            cv.resize(face,(50,50))
            if(len(data)<800):
                data.append(face)
        cv.imshow('result',frame)
        if cv.waitKey(2)==27 or len(data)>=400:
          break
video_Capture.release()
cv.destroyAllWindows()

In [16]:
np.save('With_Mask.npy',data)

In [17]:
# TO load .npy file in an Array we ise np.load function but to make this work we have to set allow_pickel = True
data_without_mask = np.load('Without_Mask.npy',allow_pickle=True)

In [18]:
data_with_mask = np.load('With_Mask.npy',allow_pickle=True)

In [19]:
data_without_mask.shape

(400,)

In [20]:
data_with_mask.shape

(400,)

In [21]:
data_without_mask =data_without_mask.reshape(100,4)

In [22]:
data_with_mask = data_with_mask.reshape(100,4)

In [23]:
X = np.r_[data_with_mask,data_without_mask]

In [24]:
Y = np.zeros(X.shape[0])
# Y is basically used as a label to show if we are wearing a mask on picture or not

In [25]:
Y[100:] = 1.0
# This statement is assigning value 1.0 to index 50 and above of  Y 

In [26]:
!pip install scikit-learn
# Statement used to install SKLEARN

In [27]:
from sklearn.svm import SVC
# HERE WE ARE IMPORTING SVC TO GET THE BEST OF X_TRAIN AND Y_TRAIN

In [28]:
from sklearn.metrics import accuracy_score
# HERE WE ARE IMPORTING ACCURACY SCORE TO CHECK THE ACCURACY

In [29]:
from sklearn.model_selection import train_test_split
# TRAIN_TEST_SPLIT IS USED TO SPLIT DATA INTO TRAINING AND TESTING DATA

In [31]:
# To split Data in ratio 1:3 (3/4 to train Data and 1/4 to test Data)
x_train, x_test, y_train, y_test = train_test_split(X,Y,test_size=0.25)

In [32]:
x_train.shape

(150, 4)

In [33]:
from sklearn.decomposition import PCA
# PCA is Proncipal Component Analysis

In [34]:
# To tranform no of columns in training data from 4 to 3
pca = PCA(n_components = 3)
x_train = pca.fit_transform(x_train)


ValueError: setting an array element with a sequence.

In [35]:
svm = SVC() 
# Here svm is object of SVC and SVC is Support Vector Classifier
# SVC is used to get the best fit 
svm.fit(x_train,y_train)

ValueError: setting an array element with a sequence.

In [ ]:
x_test = pca.transform(x_test)
# WE ARE USING PCA.TRANSFORM TO TRANSFORM TESTING DATA SAME AS TRAINING DATA
y_pred = svm.predict(x_test)
# HERE .predict is ysed to get Prediction of y(Label) by our model

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
video_Capture = cv.VideoCapture(0)
while True:
    flag,frame = video_Capture.read()
    if flag:
        faces = haar_data.detectMultiScale(frame)
        for x,y,w,h in faces:
            cv.rectangle(frame,(x,y),(x+w,y+h),(255,255,0),2)
            face = frame[y:y+h, x:x+w, :]
            cv.resize(face,(50,50))
            face = face.reshape(1,-1)
            pred = svm.predict(face)
            if (pred == 0):
                print('Mask Detected')
            else:
                print('Mask Not Detected')
        cv.imshow('result',frame)
        if cv.waitKey(2)==27:
          break
video_Capture.release()
cv.destroyAllWindows()